In [1]:
import asyncio
from functools import partial
from pathlib import Path

import ipywidgets as widgets
import pygame
from IPython.display import display
from ipyevents import Event

# Clean up when this cell is rerun.
if "intro_monitor_task" in globals():
    intro_monitor_task.cancel()

if "key_events" in globals():
    for listener in key_events:
        listener.close()

# Local audio folders
audio_dir = Path(r"D:\KAOS\visionNov132026-EDITS\audio")
tones_dir = audio_dir / "phoneTones"

# Ordered newest to oldest. Key "0" is each project's intro.
projects = [
    {
        "name": "weLoveLeimert",
        "folder": audio_dir / "weLoveLeimertArchive-Nov132026",
        "recordings": {
            "0": "noPriorArt1.wav",
            "1": "AmaraKiona.wav",
            "2": "BenCaldwell.wav",
            "3": "Benu.wav",
            "4": "HughAugustine.wav",
            "5": "Jimetta.wav",
            "6": "MaryChristian.wav",
            "7": "MattGibson.wav",
            "8": "Mitebrafoagindotan.wav",
            "9": "WeLoveLeimertAudioCollage.wav",
        },
    },
    {
        "name": "noPriorArt",
        "folder": audio_dir / "noPriorArt",
        "recordings": {
            "0": "ButtonPrompts.wav",
            "1": "VisionOfSankofaCity.wav",
            "2": "KAOSKids.wav",
            "3": "AvailableRoutes.wav",
            "4": "WhereWeAreNow.wav",
            "5": "CallTheFuture.wav",
            "6": "PreparingForOlympics.wav",
            "7": "TransportToTheFunkaverse.wav",
        },
    },
]

pygame.mixer.init()
pygame.mixer.stop()

# Load short keypad tones and project intros in advance.
tone_files = {
    **{str(number): f"DTMF-{number}.wav" for number in range(10)},
    "*": "DTMF-star.wav",
    "#": "DTMF-pound.wav",
}

key_sounds = {
    key: pygame.mixer.Sound(str(tones_dir / filename))
    for key, filename in tone_files.items()
}

intro_sounds = [
    pygame.mixer.Sound(str(project["folder"] / project["recordings"]["0"]))
    for project in projects
]

channel = pygame.mixer.Channel(0)

phone_off_hook = False
active_key = None
project_index = 0

phone_output = widgets.Output()
project_label = widgets.Label()


def update_project_label(filename=None):
    project = projects[project_index]
    playback = f"Playing: {filename}" if filename else "No audio playing"
    project_label.value = (
        f"Project {project_index}: {project['name']} | {playback}"
    )


def play_intro():
    channel.play(intro_sounds[project_index], loops=-1)
    project = projects[project_index]
    filename = project["recordings"]["0"]
    update_project_label(filename)
    print(f"Intro: {project['name']} — {filename}")


def pick_up():
    global phone_off_hook, project_index, active_key

    if phone_off_hook:
        return

    pygame.mixer.stop()
    project_index = 0
    active_key = None
    phone_off_hook = True
    print("Phone picked up.")
    play_intro()


def hang_up():
    global phone_off_hook, project_index, active_key

    phone_off_hook = False
    active_key = None
    project_index = 0
    pygame.mixer.stop()
    update_project_label()
    print("Phone hung up. Reset to project 0.")


def press_key(key):
    global active_key

    if not phone_off_hook:
        print("Pick up the phone first.")
        return

    if active_key is not None:
        return

    # Ignore boundary navigation without interrupting playback.
    if key == "#" and project_index == len(projects) - 1:
        print("Already at the oldest project.")
        return

    if key == "*" and project_index == 0:
        print("Already at the newest project.")
        return

    project = projects[project_index]

    if key not in ("*", "#") and key not in project["recordings"]:
        print(f"No recording on key {key} in {project['name']}.")
        return

    pygame.mixer.stop()
    active_key = key
    channel.play(key_sounds[key], loops=-1)
    update_project_label(tone_files[key])
    print(f"Pressed: {key}")


def release_key(key):
    global active_key, project_index

    if active_key != key:
        return

    channel.stop()
    active_key = None
    update_project_label()

    if not phone_off_hook:
        return

    if key in ("*", "#"):
        project_index += 1 if key == "#" else -1
        print(
            f"Changed to project {project_index}: "
            f"{projects[project_index]['name']}"
        )
        play_intro()
        return

    project = projects[project_index]
    filename = project["recordings"][key]
    recording = pygame.mixer.Sound(str(project["folder"] / filename))
    channel.play(recording)
    update_project_label(filename)
    print(f"Released: {key}. Playing: {project['name']} — {filename}")


def cancel_key(key):
    global active_key

    if active_key == key:
        channel.stop()
        active_key = None
        update_project_label()
        print(f"Cancelled key: {key}")
        # The monitor will resume this project's intro.


async def monitor_playback():
    while True:
        await asyncio.sleep(0.1)

        if (
            phone_off_hook
            and active_key is None
            and not channel.get_busy()
        ):
            with phone_output:
                play_intro()


# Pickup/hang-up toggle
hook_toggle = widgets.ToggleButton(
    value=False,
    description="Pick Up",
    layout=widgets.Layout(width="222px"),
)


def on_hook_change(change):
    with phone_output:
        if change["new"]:
            pick_up()
            hook_toggle.description = "Hang Up"
        else:
            hang_up()
            hook_toggle.description = "Pick Up"


hook_toggle.observe(on_hook_change, names="value")


# Mouse press/release handling
def on_key_event(event, key):
    event_type = event["type"]

    if event_type in ("mousedown", "mouseup"):
        if event.get("button", 0) != 0:
            return

    with phone_output:
        if event_type == "mousedown":
            press_key(key)
        elif event_type == "mouseup":
            release_key(key)
        elif event_type == "mouseleave":
            cancel_key(key)


# Build the keypad.
rows = []
key_events = []

for keys in [
    ["1", "2", "3"],
    ["4", "5", "6"],
    ["7", "8", "9"],
    ["*", "0", "#"],
]:
    buttons = []

    for key in keys:
        button = widgets.Button(
            description=key,
            layout=widgets.Layout(width="70px", height="50px"),
        )

        listener = Event(
            source=button,
            watched_events=["mousedown", "mouseup", "mouseleave"],
            prevent_default_action=True,
        )
        listener.on_dom_event(partial(on_key_event, key=key))
        key_events.append(listener)
        buttons.append(button)

    rows.append(widgets.HBox(buttons))

keypad = widgets.VBox(rows)

phone_controls = widgets.VBox([
    project_label,
    hook_toggle,
    keypad,
    phone_output,
])

update_project_label()
intro_monitor_task = asyncio.create_task(monitor_playback())

display(phone_controls)

with phone_output:
    print("Prototype ready. Phone is hung up.")

pygame 2.6.1 (SDL 2.28.4, Python 3.13.15)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [11]:
print("Kernel responds")
print("Phone picked up:", phone_off_hook)
print("Active key:", active_key)
print("Monitor finished:", intro_monitor_task.done())

if intro_monitor_task.done() and not intro_monitor_task.cancelled():
    print("Monitor error:", intro_monitor_task.exception())

Kernel responds
Phone picked up: True
Active key: None
Monitor finished: False


In [15]:
def log_key_event(event, key):
    with phone_output:
        print(f"EVENT: {event['type']} on {key}")


for listener, key in zip(
    key_events,
    ["1", "2", "3", "4", "5", "6", "7", "8", "9", "*", "0", "#"],
):
    listener.on_dom_event(partial(log_key_event, key=key))